# Local Corp Income Agents Notebook

In [ ]:
from __future__ import annotations

import json
import os
import sys

from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv


PROJECT_ROOT = Path().resolve()
# override=False: a value already set (e.g. from dbutils.secrets in the next
# cell) is never replaced by an empty line in a stray .env file.
load_dotenv(PROJECT_ROOT / ".env", override=False)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# LLM settings — pick ONE block, uncomment it, fill the <...>.
# A file named exactly ".env" next to this notebook works too (cell 2 reads it);
# ".env.example" is only a template and is never read.
# Values set here win over .env. Run this cell AFTER any restartPython().
import os

# --- A. OpenAI (key cá nhân / công ty) -------------------------------------
# os.environ["LLM_PROVIDER"] = "openai"
# os.environ["OPENAI_API_KEY"] = dbutils.secrets.get("<scope>", "<key-name>")   # or "sk-..." for a quick test
# os.environ["MODEL_DOCUMENT"] = "gpt-4o-mini"
# os.environ["MODEL_ECONOMY"] = "gpt-4o-mini"
# os.environ["MODEL_PREMIUM"] = "gpt-4o"

# --- B. Databricks model serving (no key of your own; data stays in the workspace)
# _ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
# os.environ["LLM_PROVIDER"] = "databricks"
# os.environ["DATABRICKS_HOST"] = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
# os.environ["DATABRICKS_TOKEN"] = _ctx.apiToken().get()
# os.environ["MODEL_DOCUMENT"] = os.environ["MODEL_ECONOMY"] = os.environ["MODEL_PREMIUM"] = "<endpoint name from Serving>"

# --- C. Anthropic ------------------------------------------------------------
# os.environ["LLM_PROVIDER"] = "anthropic"
# os.environ["ANTHROPIC_API_KEY"] = dbutils.secrets.get("<scope>", "<key-name>")
# os.environ["MODEL_DOCUMENT"] = os.environ["MODEL_ECONOMY"] = "<model>"
# os.environ["MODEL_PREMIUM"] = "<model>"

# OCR: Vietnamese only reads diacritics far better than vie+eng
os.environ.setdefault("OCR_LANG", "vie")
# os.environ["OCR_CACHE_DIR"] = "/Volumes/<catalog>/<schema>/<volume>/ocr_cache"


In [ ]:
TESTCASE_ID = "case_1"

# KHDN
LOAN_PROGRAM = "KHDN"

# Possible value: CORP_INCOME_AGENT
AGENT = "CORP_INCOME_AGENT"

# CSV/MD/PDF/TXT/XLS/XLSX/XML/DOCX/JPG/PNG — each file inside one of the upload
# box folders: ho_so_phap_ly, ho_so_tai_chinh, ho_so_vay_von,
# ho_so_tai_san_dam_bao, ho_so_noi_bo, ho_so_soan_thao_noi_bo, ho_so_khac
INPUT_PATHS = [
    str(PROJECT_ROOT / "samples" / TESTCASE_ID),
    # "/Volumes/<catalog>/<schema>/<volume>/khdn_abc",
]

# The 12-month checks end on this date (dd/mm/yyyy); None = today
LOAN_DATE = None
# KH cá nhân vay vốn. Để trống [] = tự tìm tên thành viên góp vốn / NĐDPL trong
# hồ sơ vay vốn (ho_so_vay_von, rồi ho_so_soan_thao_noi_bo). Điền tay chỉ khi muốn ghi đè.
BORROWER_NAMES = []

# Optional: BRD B2/B3/B4 — {item: JSON}, item = legal_basic | shareholders | legal_rep | liabilities
INTERNAL_DATA = {}
EXTERNAL_DATA = {}
WATCHLIST = []          # [{"name": "...", "tax_code": "...", "list": "AML"}]

# [BRD 16/10] Nghĩa vụ trả gốc TDH hằng năm của DN (đồng) — mục 6 "LNST sử dụng để trả
# nghĩa vụ TDH". None = chưa có (tính 0 và ghi chú trong báo cáo).
ANNUAL_LT_PRINCIPAL = None

OUTPUT_DIR = PROJECT_ROOT / "logs"
MAX_CHARS_PER_DOCUMENT = 120_000

In [ ]:
from src.config import Config, build_llm

config = Config(
    document_llm=build_llm(
        "MODEL_DOCUMENT",
        temperature=0.5
    ),
    legal_basic_extraction_llm=build_llm(
        "MODEL_PREMIUM",
        temperature=0.0,
    ),
    shareholders_extraction_llm=build_llm(
        "MODEL_PREMIUM",
        temperature=0.0,
    ),
    legal_rep_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    liabilities_extraction_llm=build_llm(
        "MODEL_ECONOMY",
        temperature=0.0,
    ),
    # Phụ lục 01 mục 3: số liệu BCTC (tỷ số tính bằng code, không do LLM)
    financial_extraction_llm=build_llm(
        "MODEL_PREMIUM",
        temperature=0.0,
    ),
    # Viết các ô nhận xét / HĐKD / CIC / kết luận theo guidance
    # Văn phong: 0.2 cho câu chữ tự nhiên hơn mà kết luận vẫn ổn định giữa các lần chạy.
    # Không nên > 0.3 cho tờ trình tín dụng. Model mạnh hơn (Sonnet) cải thiện nhiều hơn temperature.
    report_writer_llm=build_llm(
        "MODEL_PREMIUM",
        temperature=0.2,
    ),
    # [BRD 16/10] tờ khai VAT / CIC / phương án kinh doanh: để trống thì dùng
    # financial_extraction_llm. Muốn model riêng thì thêm, ví dụ:
    # cic_extraction_llm=build_llm("MODEL_PREMIUM", temperature=0.0),
    # business_model_extraction_llm=build_llm("MODEL_ECONOMY", temperature=0.0),  # [BRD 27/10] mục 2
    max_chars_per_document=MAX_CHARS_PER_DOCUMENT,
)

In [ ]:
# Check the models BEFORE the run: one tiny call per model.
# Stops "Run all" here with the real cause (key, URL, firewall, quota…)
# instead of failing every extraction after the OCR.
from src.llm_check import check_llm_setup

check_llm_setup(config)


In [ ]:
from src.types import (
    AgentName,
    WorkflowMode,
    DocumentAgentName,
    ClassifiedDocument,
    UnderwritingGraphState,
    to_dict_list,
    extract_text_from_agent_output,
    truncate_text,
)

In [ ]:
from src.agents.documents.document_classification import (
    document_type_scores,
    rule_classify_document,
)
from src.agents.documents.document_discovery import (
    compute_file_hash,
    discover_documents,
    resolve_input_path,
)
from src.utils.common import (
    SUPPORTED_EXTENSIONS,
    normalize_text,
)
from src.types import VALID_DOCUMENT_AGENTS
from src.agents.documents.document_matrix import (
    agent_relevance_for_type,
    load_matrix,
    primary_agent_for_type,
)

_matrix = load_matrix()
print(
    f"Document matrix v{_matrix.version}: {len(_matrix.types)} document types \n"
    f"Loan programs: {', '.join(_matrix.loan_programs)}"
)

In [ ]:
from src.agents.specialist import (
    CorpIncomeAnalysis,
)
from src.agents.supervisor import Supervisor

In [ ]:
from IPython.display import Markdown, display
from src.utils.common import show_graph
from src.agents.supervisor import EXTRACTION_PASSES

supervisor = Supervisor(config)

display(Markdown("### Workflow Graph"))
display(show_graph(supervisor.workflow_graph))

result = supervisor.process(
    INPUT_PATHS,
    agent=AGENT,
    loan_program=LOAN_PROGRAM,
    loan_date=LOAN_DATE,
    borrower_names=BORROWER_NAMES,
    internal_data=INTERNAL_DATA,
    external_data=EXTERNAL_DATA,
    watchlist=WATCHLIST,
    annual_lt_principal=ANNUAL_LT_PRINCIPAL,
)

display(Markdown(result["response"]))

print("\n--- Agent Name ---")
print(result["agent_name"])

print("\n--- Steps ---")
for step in result["steps"]:
    print("-", step)

print("\n--- Document Classifications ---")
classification_keys = [
    "filename",
    "document_type",
    "declared_group",
    "document_group",
    "source",
    "source_description",
    "read_method",
    "agent_relevance",
    "loan_program",
    "confidence",
    "reasoning",
    "extraction_status",
    "extraction_error",
    "is_legal_basic",
    "legal_basic_extraction_error",
    "is_shareholders",
    "shareholders_extraction_error",
    "is_legal_rep",
    "legal_rep_extraction_error",
    "is_liabilities",
    "liabilities_extraction_error",
    "is_financial",
    "financial_extraction_error",
    "is_vat",
    "vat_extraction_error",
    "is_cic",
    "cic_extraction_error",
    "is_plan",
    "plan_extraction_error",
    "is_business_model",
    "business_model_extraction_error",
    "classifier_error_type",
    "classifier_error",
]
for item in result["document_classifications"]:
    classification = {key: item.get(key) for key in classification_keys}
    print(json.dumps(classification, ensure_ascii=False, indent=2))

_failures = [
    (item["filename"], pass_, item.get(pass_.error_attr) or "Không rõ lý do.")
    for item in result["document_classifications"]
    for pass_ in EXTRACTION_PASSES
    if item.get(pass_.flag_attr) and not item.get(pass_.result_attr)
]
if _failures:
    print(f"\nWARNING: {len(_failures)} lần trích xuất thất bại:")
    for _name, _pass, _why in _failures:
        print(f"  - {_name} ({_pass.label}): {_why}")

_unmatched = [
    item["filename"]
    for item in result["document_classifications"]
    if not item.get("document_type")
]
if _unmatched:
    print(
        f"\nWARNING: {len(_unmatched)} document(s) matched no type in the "
        f"matrix and were kept as general context: {', '.join(_unmatched)}"
    )

print("\n--- Corp Income ---")
for _n, _item in (result.get("corp_income", {}).get("items") or {}).items():
    print(f"{_n}. {_item['title']}: {_item['conclusion']}")

In [ ]:
run_id = "_".join([TESTCASE_ID, datetime.now().strftime("%Y%m%d_%H%M%S")])
run_dir = OUTPUT_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=True)

from src.utils.report.visualization.report_html import embed_diagrams
from src.utils.report.excel_export import write_excel

post_monitor_content = [
    {
        'filename': 'result.json',
        'content': result
    },
    {
        'filename': 'document_classifications.json',
        'content': result["document_classifications"]
    },
    {
        'filename': 'corp_income.json',
        'content': result["corp_income"]
    },
    {
        'filename': 'legal_basic_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'legal_basic_extraction',
        'condition': 'is_legal_basic'
    },
    {
        'filename': 'shareholders_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'shareholders_extraction',
        'condition': 'is_shareholders'
    },
    {
        'filename': 'legal_rep_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'legal_rep_extraction',
        'condition': 'is_legal_rep'
    },
    {
        'filename': 'liabilities_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'liabilities_extraction',
        'condition': 'is_liabilities'
    },
    {
        'filename': 'financial_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'financial_extraction',
        'condition': 'is_financial'
    },
    # [BRD 16/10]
    {
        'filename': 'vat_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'vat_extraction',
        'condition': 'is_vat'
    },
    {
        'filename': 'cic_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'cic_extraction',
        'condition': 'is_cic'
    },
    {
        'filename': 'plan_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'plan_extraction',
        'condition': 'is_plan'
    },
    # [BRD 27/10]
    {
        'filename': 'business_model_extraction.json',
        'content': result["document_classifications"],
        'content_lv2': 'business_model_extraction',
        'condition': 'is_business_model'
    },
]

(run_dir / "final_response.md").write_text(
    embed_diagrams(result["response"]),
    encoding="utf-8"
)

_own_file_keys = {
    c['filename'].removesuffix('.json')
    for c in post_monitor_content
    if not c.get('content_lv2') and c['filename'] != 'result.json'
}
_own_file_doc_keys = {
    c['content_lv2'] for c in post_monitor_content if c.get('content_lv2')
}

for c in post_monitor_content:
    if c['filename'] == 'result.json':
        continue
    if c.get('content_lv2', None):
        matched = [doc for doc in c['content'] if doc.get(c['condition'])]
        payload = {
            doc["filename"]: doc.get(c['content_lv2']) for doc in matched
        }
        (run_dir / c['filename']).write_text(
            json.dumps(payload, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
    else:
        (run_dir / c['filename']).write_text(
            json.dumps(c['content'], ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

# Full OCR / text of every file — check here first when an extracted field is wrong.
(run_dir / "ocr").mkdir(exist_ok=True)
for _filename, _text in (result.get("ocr_text") or {}).items():
    (run_dir / "ocr" / f"{_filename}.txt").write_text(_text, encoding="utf-8")

# The BRD layout as a workbook: one sheet per item + findings + files.
if result.get("corp_income"):
    write_excel(result["corp_income"], run_dir / "corp_income.xlsx")

_slim_result = {
    key: value
    for key, value in result.items()
    if key not in _own_file_keys and key not in ('document_classifications', 'ocr_text')
}
_slim_result['document_classifications'] = [
    {k: v for k, v in doc.items() if k not in _own_file_doc_keys}
    for doc in result['document_classifications']
]
_slim_result['artifacts'] = (
    'Extraction results, the corp_income result and document classifications are '
    'in the sibling JSON files; full OCR text is in ocr/; corp_income.xlsx has the '
    'BRD layout.'
)
(run_dir / 'result.json').write_text(
    json.dumps(_slim_result, ensure_ascii=False, indent=2), encoding='utf-8'
)

# Export the final response (Markdown) to PDF via `markdown` + WeasyPrint.
import platform

if platform.system() == "Darwin":
    for brew_lib in ("/opt/homebrew/lib", "/usr/local/lib"):
        if os.path.isdir(brew_lib):
            os.environ["DYLD_LIBRARY_PATH"] = (
                brew_lib + ":" + os.environ.get("DYLD_LIBRARY_PATH", "")
            )

try:
    from weasyprint import HTML
    from src.utils.report.visualization.report_html import build_report_html

    html_doc = build_report_html(result["response"])
    HTML(string=html_doc).write_pdf(str(run_dir / "final_response.pdf"))
    print(f"Saved PDF: {run_dir / 'final_response.pdf'}")
except (ImportError, OSError) as exc:
    # WeasyPrint needs Pango/Cairo system libraries; the .md and .xlsx are
    # already saved, so a missing PDF engine must not lose the run.
    print(f"Skipped PDF export ({type(exc).__name__}: {exc}). The .md and .xlsx are saved.")

print(f"Saved artifacts to: {run_dir}")